
## Folder 07 — File 06: Gold Inventory Status:


This notebook creates inventory analytics using:
- silver_inventory
- silver_products


It calculates:

- Current stock by product.
- Current stock by warehouse.
- Low-stock and out-of-stock products.
- Inventory stock valuation.
- Inventory availability percentage.
- Historical inventory snapshot trends.
- Products with inventory but no recent snapshots.
- Warehouse-level inventory summaries.

**Important**: 

- Inventory snapshots represent stock at a point in time. 
- They are not individual stock movement transactions. 
- We can calculate changes between snapshots, but cannot reliably identify purchases, sales, returns or adjustments without a movement dataset.

In [0]:
## 1. Import libraries

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

print("Libraries imported successfully.")


## 2. Project configuration

CATALOG = "workspace"

SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

INVENTORY_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_inventory"
)

PRODUCTS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.silver_products"
)

GOLD_INVENTORY_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_inventory_status"
)

GOLD_WAREHOUSE_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_warehouse_inventory"
)

GOLD_SNAPSHOT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_inventory_snapshot_trends"
)

GOLD_ALERT_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}.gold_inventory_alerts"
)

RUN_ID = spark.sql(
    "SELECT uuid() AS run_id"
).first()["run_id"]

RUN_TIMESTAMP = datetime.now()

print("Run ID:", RUN_ID)


## 3. Validate source tables

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}"
)

SOURCE_TABLES = [
    INVENTORY_TABLE,
    PRODUCTS_TABLE
]

for table_name in SOURCE_TABLES:

    if not spark.catalog.tableExists(table_name):

        raise Exception(
            f"Missing source table: {table_name}"
        )

print("All source tables are available.")


## 4. Load Silver datasets

inventory_df = spark.table(INVENTORY_TABLE)

products_df = spark.table(PRODUCTS_TABLE)

print("Inventory records:", inventory_df.count())
print("Product records:", products_df.count())


## 5. Validate required columns

required_inventory_columns = [
    "inventory_snapshot_id",
    "product_id",
    "quantity_on_hand",
    "snapshot_date",
    "warehouse_id"
]

required_product_columns = [
    "product_id",
    "product_name",
    "category",
    "unit_price",
    "unit_cost",
    "stock_status"
]

for column_name in required_inventory_columns:

    if column_name not in inventory_df.columns:

        raise Exception(
            f"Missing inventory column: {column_name}"
        )

for column_name in required_product_columns:

    if column_name not in products_df.columns:

        raise Exception(
            f"Missing product column: {column_name}"
        )


## 6. Clean inventory records

inventory_clean_df = (
    inventory_df

    .withColumn(
        "inventory_snapshot_id",
        F.trim(F.col("inventory_snapshot_id"))
    )

    .withColumn(
        "product_id",
        F.trim(F.col("product_id"))
    )

    .withColumn(
        "warehouse_id",
        F.upper(F.trim(F.col("warehouse_id")))
    )

    .withColumn(
        "quantity_on_hand",
        F.col("quantity_on_hand").cast("int")
    )

    .withColumn(
        "snapshot_date",
        F.to_date("snapshot_date")
    )

    .filter(
        F.col("inventory_snapshot_id").isNotNull()
        & (F.col("inventory_snapshot_id") != "")
        & F.col("product_id").isNotNull()
        & (F.col("product_id") != "")
        & F.col("warehouse_id").isNotNull()
        & (F.col("warehouse_id") != "")
        & F.col("quantity_on_hand").isNotNull()
        & (F.col("quantity_on_hand") >= 0)
        & F.col("snapshot_date").isNotNull()
    )
)


## 7. Deduplicate inventory snapshot records

# The inventory business key is:
# product_id + warehouse_id + snapshot_date.
# If duplicate records exist for this key, retain the latest ingested record.

if "_ingestion_timestamp" in inventory_clean_df.columns:

    inventory_window = (
        Window

        .partitionBy(
            "product_id",
            "warehouse_id",
            "snapshot_date"
        )

        .orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last(),
            F.col("inventory_snapshot_id").desc()
        )
    )

else:

    inventory_window = (
        Window

        .partitionBy(
            "product_id",
            "warehouse_id",
            "snapshot_date"
        )

        .orderBy(
            F.col("inventory_snapshot_id").desc()
        )
    )

inventory_deduplicated_df = (
    inventory_clean_df

    .withColumn(
        "_row_number",
        F.row_number().over(inventory_window)
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop("_row_number")
)


## 8. Clean product data

products_clean_df = (
    products_df

    .withColumn(
        "product_id",
        F.trim(F.col("product_id"))
    )

    .withColumn(
        "product_name",
        F.trim(F.col("product_name"))
    )

    .withColumn(
        "category",
        F.initcap(F.trim(F.col("category")))
    )

    .withColumn(
        "stock_status",
        F.upper(F.trim(F.col("stock_status")))
    )

    .withColumn(
        "unit_price",
        F.col("unit_price").cast("decimal(18,2)")
    )

    .withColumn(
        "unit_cost",
        F.col("unit_cost").cast("decimal(18,2)")
    )

    .filter(
        F.col("product_id").isNotNull()
        & (F.col("product_id") != "")
        & F.col("product_name").isNotNull()
    )
)


## 9. Deduplicate products

if "_ingestion_timestamp" in products_clean_df.columns:

    product_window = (
        Window

        .partitionBy("product_id")

        .orderBy(
            F.col("_ingestion_timestamp").desc_nulls_last()
        )
    )

else:

    product_window = (
        Window

        .partitionBy("product_id")

        .orderBy(
            F.col("product_name").desc()
        )
    )

products_deduplicated_df = (
    products_clean_df

    .withColumn(
        "_row_number",
        F.row_number().over(product_window)
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop("_row_number")
)


## 10. Identify latest inventory snapshot per product and warehouse

# Do not sum historical snapshots to calculate current stock.
# Select the latest snapshot for each product and warehouse.


latest_inventory_window = (
    Window

    .partitionBy(
        "product_id",
        "warehouse_id"
    )

    .orderBy(
        F.col("snapshot_date").desc()
    )
)

latest_inventory_df = (
    inventory_deduplicated_df

    .withColumn(
        "_row_number",
        F.row_number().over(latest_inventory_window)
    )

    .filter(
        F.col("_row_number") == 1
    )

    .drop("_row_number")
)

print(
    "Latest product-warehouse inventory records:",
    latest_inventory_df.count()
)


## 11. Calculate current stock by product

# Aggregate the latest warehouse snapshots for each product.

current_product_inventory_df = (
    latest_inventory_df

    .groupBy(
        "product_id"
    )

    .agg(

        F.sum("quantity_on_hand").alias(
            "total_quantity_on_hand"
        ),

        F.countDistinct("warehouse_id").alias(
            "warehouse_count"
        ),

        F.max("snapshot_date").alias(
            "latest_snapshot_date"
        ),

        F.sum(
            F.when(
                F.col("quantity_on_hand") == 0,
                1
            ).otherwise(0)
        ).alias(
            "out_of_stock_warehouse_count"
        ),

        F.sum(
            F.when(
                F.col("quantity_on_hand") > 0,
                1
            ).otherwise(0)
        ).alias(
            "available_warehouse_count"
        )
    )
)


## 12. Join product information with current inventory

gold_inventory_df = (
    products_deduplicated_df.alias("products")

    .join(
        current_product_inventory_df.alias("inventory"),

        F.col("products.product_id")
        == F.col("inventory.product_id"),

        "left"
    )

    .select(

        F.col("products.product_id"),

        F.col("products.product_name"),

        F.col("products.category"),

        F.col("products.stock_status"),

        F.col("products.unit_price"),

        F.col("products.unit_cost"),

        F.coalesce(
            F.col("inventory.total_quantity_on_hand"),
            F.lit(0)
        ).alias(
            "total_quantity_on_hand"
        ),

        F.coalesce(
            F.col("inventory.warehouse_count"),
            F.lit(0)
        ).alias(
            "warehouse_count"
        ),

        F.col("inventory.latest_snapshot_date"),

        F.coalesce(
            F.col("inventory.out_of_stock_warehouse_count"),
            F.lit(0)
        ).alias(
            "out_of_stock_warehouse_count"
        ),

        F.coalesce(
            F.col("inventory.available_warehouse_count"),
            F.lit(0)
        ).alias(
            "available_warehouse_count"
        )
    )
)


## 13. Calculate stock valuation

# Inventory cost valuation = quantity on hand multiplied by unit cost.
# Inventory retail valuation = quantity on hand multiplied by unit price.
# These are estimates based on the current product master prices.

gold_inventory_df = (
    gold_inventory_df

    .withColumn(
        "inventory_cost_value",

        F.when(
            F.col("unit_cost").isNotNull(),

            F.round(
                F.col("total_quantity_on_hand")
                * F.col("unit_cost"),
                2
            )
        )
    )

    .withColumn(
        "inventory_retail_value",

        F.when(
            F.col("unit_price").isNotNull(),

            F.round(
                F.col("total_quantity_on_hand")
                * F.col("unit_price"),
                2
            )
        )
    )

    .withColumn(
        "potential_gross_margin_value",

        F.when(
            F.col("inventory_cost_value").isNotNull()
            & F.col("inventory_retail_value").isNotNull(),

            F.round(
                F.col("inventory_retail_value")
                - F.col("inventory_cost_value"),
                2
            )
        )
    )
)


## 14. Create inventory stock categories

# Low-stock threshold is configurable.
# The threshold is a demonstration rule, not a business-approved reorder point.

LOW_STOCK_THRESHOLD = 10

gold_inventory_df = (
    gold_inventory_df

    .withColumn(
        "inventory_category",

        F.when(
            F.col("total_quantity_on_hand") == 0,
            "OUT_OF_STOCK"
        )

        .when(
            F.col("total_quantity_on_hand")
            <= LOW_STOCK_THRESHOLD,
            "LOW_STOCK"
        )

        .otherwise("IN_STOCK")
    )

    .withColumn(
        "inventory_availability",

        F.when(
            F.col("total_quantity_on_hand") > 0,
            "AVAILABLE"
        )

        .otherwise("UNAVAILABLE")
    )
)


## 15. Calculate inventory availability metrics

inventory_summary_df = (
    gold_inventory_df

    .groupBy(
        "category"
    )

    .agg(

        F.countDistinct("product_id").alias(
            "total_products"
        ),

        F.sum("total_quantity_on_hand").alias(
            "total_stock_quantity"
        ),

        F.sum(
            F.when(
                F.col("inventory_category") == "IN_STOCK",
                1
            ).otherwise(0)
        ).alias(
            "in_stock_products"
        ),

        F.sum(
            F.when(
                F.col("inventory_category") == "LOW_STOCK",
                1
            ).otherwise(0)
        ).alias(
            "low_stock_products"
        ),

        F.sum(
            F.when(
                F.col("inventory_category") == "OUT_OF_STOCK",
                1
            ).otherwise(0)
        ).alias(
            "out_of_stock_products"
        ),

        F.sum("inventory_cost_value").alias(
            "total_inventory_cost_value"
        ),

        F.sum("inventory_retail_value").alias(
            "total_inventory_retail_value"
        )
    )

    .withColumn(
        "product_availability_rate",

        F.when(
            F.col("total_products") > 0,

            F.round(
                (
                    F.col("in_stock_products")
                    + F.col("low_stock_products")
                )
                / F.col("total_products") * 100,
                2
            )
        ).otherwise(F.lit(0))
    )
)


## 16. Create warehouse inventory summary

#  Use the latest snapshot for each product and warehouse.
#  Do not aggregate all historical snapshot records.

warehouse_inventory_df = (
    latest_inventory_df.alias("inventory")

    .join(
        products_deduplicated_df.alias("products"),

        F.col("inventory.product_id")
        == F.col("products.product_id"),

        "left"
    )

    .groupBy(
        F.col("inventory.warehouse_id").alias("warehouse_id")
    )

    .agg(

        F.countDistinct(
            F.col("inventory.product_id")
        ).alias(
            "total_products"
        ),

        F.sum(
            F.col("inventory.quantity_on_hand")
        ).alias(
            "total_quantity_on_hand"
        ),

        F.sum(
            F.when(
                F.col("inventory.quantity_on_hand") == 0,
                1
            ).otherwise(0)
        ).alias(
            "out_of_stock_product_count"
        ),

        F.sum(
            F.when(
                F.col("inventory.quantity_on_hand")
                <= LOW_STOCK_THRESHOLD,
                1
            ).otherwise(0)
        ).alias(
            "low_stock_product_count"
        ),

        F.sum(
            F.col("inventory.quantity_on_hand")
            * F.col("products.unit_cost")
        ).alias(
            "inventory_cost_value"
        ),

        F.max(
            F.col("inventory.snapshot_date")
        ).alias(
            "latest_snapshot_date"
        )
    )
)


## 17. Create historical inventory snapshot trends

# Retain daily snapshot history.
# Calculate stock changes against the previous available snapshot for each product and warehouse.

snapshot_window = (
    Window

    .partitionBy(
        "product_id",
        "warehouse_id"
    )

    .orderBy(
        "snapshot_date"
    )
)

inventory_snapshot_trends_df = (
    inventory_deduplicated_df

    .withColumn(
        "previous_quantity",

        F.lag("quantity_on_hand").over(
            snapshot_window
        )
    )

    .withColumn(
        "quantity_change",

        F.when(
            F.col("previous_quantity").isNotNull(),

            F.col("quantity_on_hand")
            - F.col("previous_quantity")
        )
    )

    .withColumn(
        "quantity_change_type",

        F.when(
            F.col("previous_quantity").isNull(),
            "FIRST_SNAPSHOT"
        )

        .when(
            F.col("quantity_change") > 0,
            "STOCK_INCREASE"
        )

        .when(
            F.col("quantity_change") < 0,
            "STOCK_DECREASE"
        )

        .otherwise("NO_CHANGE")
    )
)


## 18. Create inventory alerts

# Generate alerts for low stock, out-of-stock products,
# missing inventory snapshots and missing product costs.

inventory_alerts_df = (
    gold_inventory_df

    .withColumn(
        "alert_type",

        F.when(
            F.col("total_quantity_on_hand") == 0,
            "OUT_OF_STOCK"
        )

        .when(
            F.col("total_quantity_on_hand")
            <= LOW_STOCK_THRESHOLD,
            "LOW_STOCK"
        )

        .when(
            F.col("latest_snapshot_date").isNull(),
            "MISSING_INVENTORY_SNAPSHOT"
        )

        .when(
            F.col("unit_cost").isNull(),
            "MISSING_UNIT_COST"
        )
    )

    .filter(
        F.col("alert_type").isNotNull()
    )

    .withColumn(
        "alert_priority",

        F.when(
            F.col("alert_type") == "OUT_OF_STOCK",
            "HIGH"
        )

        .when(
            F.col("alert_type") == "LOW_STOCK",
            "MEDIUM"
        )

        .otherwise("LOW")
    )

    .withColumn(
        "alert_generated_at",
        F.lit(RUN_TIMESTAMP)
    )
)


## 19. Add audit columns

gold_inventory_df = (
    gold_inventory_df

    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)

warehouse_inventory_df = (
    warehouse_inventory_df

    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)

inventory_snapshot_trends_df = (
    inventory_snapshot_trends_df

    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )

    .withColumn(
        "created_at",
        F.lit(RUN_TIMESTAMP)
    )
)

inventory_alerts_df = (
    inventory_alerts_df

    .withColumn(
        "run_id",
        F.lit(RUN_ID)
    )
)


## 20. Display current inventory status

display(
    gold_inventory_df.orderBy(
        F.col("total_quantity_on_hand").asc()
    )
)


# 21. Display warehouse inventory

display(
    warehouse_inventory_df.orderBy(
        F.col("total_quantity_on_hand").desc()
    )
)


## 22. Display inventory snapshot trends

display(
    inventory_snapshot_trends_df.orderBy(
        F.col("snapshot_date").desc()
    )
)


## 23. Display inventory alerts

display(
    inventory_alerts_df.orderBy(
        F.col("alert_priority"),
        F.col("total_quantity_on_hand").asc()
    )
)


## 24. Save Gold inventory status

(
    gold_inventory_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        GOLD_INVENTORY_TABLE
    )
)

print(
    "Gold inventory status created:",
    GOLD_INVENTORY_TABLE
)


## 25. Save Gold warehouse inventory

(
    warehouse_inventory_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        GOLD_WAREHOUSE_TABLE
    )
)

print(
    "Gold warehouse inventory created:",
    GOLD_WAREHOUSE_TABLE
)


## 26. Save Gold inventory snapshot trends

(
    inventory_snapshot_trends_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        GOLD_SNAPSHOT_TABLE
    )
)

print(
    "Gold inventory snapshot trends created:",
    GOLD_SNAPSHOT_TABLE
)


## 27. Save Gold inventory alerts

(
    inventory_alerts_df.write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        GOLD_ALERT_TABLE
    )
)

print(
    "Gold inventory alerts created:",
    GOLD_ALERT_TABLE
)


## 28. Final validation

gold_inventory_check_df = spark.table(
    GOLD_INVENTORY_TABLE
)

gold_warehouse_check_df = spark.table(
    GOLD_WAREHOUSE_TABLE
)

gold_snapshot_check_df = spark.table(
    GOLD_SNAPSHOT_TABLE
)

gold_alert_check_df = spark.table(
    GOLD_ALERT_TABLE
)

print(
    "Inventory product records:",
    gold_inventory_check_df.count()
)

print(
    "Warehouse records:",
    gold_warehouse_check_df.count()
)

print(
    "Inventory snapshot records:",
    gold_snapshot_check_df.count()
)

print(
    "Inventory alert records:",
    gold_alert_check_df.count()
)

display(
    gold_inventory_check_df

    .groupBy("inventory_category")

    .agg(

        F.countDistinct("product_id").alias(
            "product_count"
        ),

        F.sum("total_quantity_on_hand").alias(
            "total_quantity"
        ),

        F.sum("inventory_cost_value").alias(
            "inventory_cost_value"
        ),

        F.sum("inventory_retail_value").alias(
            "inventory_retail_value"
        )
    )
)


print("=" * 65)
print("GOLD INVENTORY STATUS COMPLETED")
print("=" * 65)

print("Inventory status:", GOLD_INVENTORY_TABLE)
print("Warehouse inventory:", GOLD_WAREHOUSE_TABLE)
print("Snapshot trends:", GOLD_SNAPSHOT_TABLE)
print("Inventory alerts:", GOLD_ALERT_TABLE)



## Databricks SQL dashboard queries:


### Current inventory overview

In [0]:
%sql

SELECT
    inventory_category,
    COUNT(*) AS total_products,
    SUM(total_quantity_on_hand) AS total_stock,
    SUM(inventory_cost_value) AS inventory_cost_value,
    SUM(inventory_retail_value) AS inventory_retail_value
FROM workspace.ecommerce_gold.gold_inventory_status
GROUP BY inventory_category;


### Low-stock products

In [0]:
%sql

SELECT
    product_id,
    product_name,
    category,
    total_quantity_on_hand,
    warehouse_count,
    inventory_category
FROM workspace.ecommerce_gold.gold_inventory_status
WHERE inventory_category = 'LOW_STOCK'
ORDER BY total_quantity_on_hand;


### Out-of-stock products

In [0]:
%sql

SELECT
    product_id,
    product_name,
    category,
    total_quantity_on_hand
FROM workspace.ecommerce_gold.gold_inventory_status
WHERE inventory_category = 'OUT_OF_STOCK';


### Warehouse stock comparison

In [0]:
%sql

SELECT
    warehouse_id,
    total_products,
    total_quantity_on_hand,
    out_of_stock_product_count,
    low_stock_product_count,
    inventory_cost_value
FROM workspace.ecommerce_gold.gold_warehouse_inventory
ORDER BY total_quantity_on_hand DESC;


### Historical stock changes

In [0]:
%sql

SELECT
    product_id,
    warehouse_id,
    snapshot_date,
    previous_quantity,
    quantity_on_hand,
    quantity_change,
    quantity_change_type
FROM workspace.ecommerce_gold.gold_inventory_snapshot_trends
ORDER BY product_id, warehouse_id, snapshot_date;

In [0]:
%sql
DESCRIBE TABLE workspace.ecommerce_gold.gold_inventory_status;